In [33]:
import pandas as pd
import xml.etree.ElementTree as ET
import os

In [34]:
#
# Define constants
#
PIPELINE_CSV_PATH   = "pipelines_20260619_143449.csv"
PIPELINE_DUMP_OUPUT = "jenkinsDump_2026_06_19_15_05_51_edited\ESB"
PIPELINE_TO_PROCESS = "jenkinsDump_2026_06_19_15_05_51\ESB"

<>:5: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:6: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:5: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:6: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
C:\Users\g.luppina\AppData\Local\Temp\ipykernel_30020\4626052.py:5: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
  PIPELINE_DUMP_OUPUT = "jenkinsDump_2026_06_19_15_05_51_edited\ESB"
C:\Users\g.luppina\AppData\Local\Temp\ipykernel_30020\4626052.py:6: SyntaxWarning: "\E" is an invalid es

In [35]:
# Read pipelines from disk and select all those associated with ESB
pipelines = pd.read_csv(PIPELINE_CSV_PATH)
pipelines = pipelines[pipelines["FOLDER_NAME"] == "ESB"]

In [36]:
os.makedirs(PIPELINE_DUMP_OUPUT, exist_ok=True)

for xml in os.listdir(PIPELINE_TO_PROCESS):
    if not xml.endswith('.xml'):
        continue
    
    name = xml.replace(".xml", "")
    path = os.path.join(PIPELINE_TO_PROCESS, xml)
    tree = ET.parse(path)
    root = tree.getroot()
    
    records = pipelines[pipelines['PIPELINE_NAME'] == name]
    if len(records) != 1:
        continue
    
    #
    # Get the record associate with current XML and debug the
    # content
    #
    settings = records.iloc[0]
    # for col in settings.index:
    #         value = settings[col]
    #         print(f"  {col}: {value!r} (type: {type(value).__name__})")
    # print()
    
    #
    # Get parameters definition
    #
    param_def_property = root.find('.//hudson.model.ParametersDefinitionProperty')
    if param_def_property is None:
        continue
    param_definitions = param_def_property.find('parameterDefinitions')
    
    #
    # Remove any hidden parameter
    #
    for hidden_param in param_definitions.findall('com.wangyin.parameter.WHideParameterDefinition'):
        name_elem = hidden_param.find('name')
        if name_elem is not None and name_elem.text == 'applicationCode':
            param_definitions.remove(hidden_param)
    
    #
    # Remove any visible parameter
    #
    for child in list(param_definitions):
        param_definitions.remove(child)
    
    # Determine configuration flags
    has_jvm  = settings.get("JVMS", False) == True
    
    # 
    # If not using Jenkinsfile as script
    #
    if not settings.get("SCM_JENKINSFILE", "") == "Jenkinsfile":
        jdks = settings.get("JDKS", "")
        
        # Handle JDKS: must be a string with space-separated values
        if isinstance(jdks, str) and jdks.strip():
            jdk_choices = [x.strip() for x in jdks.split()]
            
            if jdk_choices:
                jdk_param = ET.Element('hudson.model.ChoiceParameterDefinition')
                ET.SubElement(jdk_param, 'name').text = 'jdkVersion'
                ET.SubElement(jdk_param, 'description').text = ''
                
                choices_elem = ET.SubElement(jdk_param, 'choices')
                choices_elem.set('class', 'java.util.Arrays$ArrayList')
                string_array = ET.SubElement(choices_elem, 'a')
                string_array.set('class', 'string-array')
                
                for jdk_choice in jdk_choices:
                    ET.SubElement(string_array, 'string').text = jdk_choice
                
                param_definitions.append(jdk_param)     
    
    #
    # Add remaining parmaters
    #
    pipeline_parts = settings["PIPELINE_NAME"].rsplit('-', 1)
    params_to_add = [
        ('gitlabUrl'        , 'Url Gitlab'              , 'http://172.16.63.237'),
        ('gitlabGroup'      , 'Nome del gruppo Gitlab'  , settings["FOLDER_NAME"]),
        ('gitlabPproject'   , 'Nome del progetto gitlab', pipeline_parts[0]),
        ('gitlabBranch'     , 'Nome della branch'       , pipeline_parts[1]),
    ]
    
    #
    # Add such parameters and then JVMAdditionalParameters if and if only
    # using different JDKs
    # 
    #if has_jdk:
    params_to_add.extend([
            ('mvnBuild'               , '', 'clean compile'),
            ('mvnUIT'                 , '', 'verify'),
            ('mvnArtifactDeploy'      , '', 'install'),
            ('JVMAdditionalParameters', '', ''),
        ])
    
    # Add all parameters
    for param_name, param_desc, default_value in params_to_add:
        if param_name == "JVMAdditionalParameters":
            if settings.get("JVMS", False) == False:
                continue
        new_param = ET.Element('hudson.model.StringParameterDefinition')
        ET.SubElement(new_param, 'name').text           = param_name
        ET.SubElement(new_param, 'description').text    = param_desc
        ET.SubElement(new_param, 'defaultValue').text   = default_value
        ET.SubElement(new_param, 'trim').text = 'false'
        param_definitions.append(new_param)
        
    # Update Gitlab URL (checkout)
    for user_remote_config in root.findall('.//hudson.plugins.git.UserRemoteConfig'):
        url_elem = user_remote_config.find('url')
        if url_elem is not None:
            url_elem.text = settings["SCM_REPOSITORY"]
    
    # Update Gitlab branch (checkout)
    for branch_spec in root.findall('.//hudson.plugins.git.BranchSpec'):
        name_elem = branch_spec.find('name')
        if name_elem is not None:
            name_elem.text = settings["SCM_BRANCH"]
    
    # Update Gitlab script
    script_path_elem = root.find('.//scriptPath')
    if script_path_elem is not None:
        script_path_elem.text = settings["SCM_JENKINSFILE"]
    
    # Write output
    output_path = os.path.join(PIPELINE_DUMP_OUPUT, xml)
    tree.write(output_path, encoding='utf-8', xml_declaration=True)